# 1) Job Type Trends and Work-Type Composition in NYC Filings

In [1]:
import os
import sys
import json
from pathlib import Path


import pandas as pd
import numpy as np
import altair as alt


import geopandas as gpd
from shapely.geometry import Point

# Show all rows
pd.set_option('display.max_rows', None)

# Show all columns
pd.set_option('display.max_columns', None)
#pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
#pd.set_option("display.max_colwidth", 200)

print("Versions ->",
      "pandas:", pd.__version__,
      "| geopandas:", gpd.__version__)

Versions -> pandas: 2.2.2 | geopandas: 1.1.1


In [2]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [3]:
file_path = '/content/drive/MyDrive/CS424-Assignment3/df_sub.pkl'
DF = pd.read_pickle(file_path)



In [5]:
DF.dtypes

,0
Job Filing Number,object
Filing Status,object
House No,object
Street Name,object
Borough,object
Block,int64
LOT,int64
Bin,int64
Commmunity - Board,float64
Work on Floor,object


In [ ]:
# --Clustered bars with interactive legend highlight and brushable overview + rolling mean ---

alt.data_transformers.disable_max_rows()


src = DF[["Filing Date", "Job Type"]].dropna(subset=["Filing Date", "Job Type"]).copy()

# Last FULL month
data_max = src["Filing Date"].max()
this_month_start = pd.Timestamp(year=data_max.year, month=data_max.month, day=1)
end_of_this_month = this_month_start + pd.offsets.MonthEnd(1)
last_full_month = this_month_start if data_max >= end_of_this_month else (this_month_start - pd.DateOffset(months=1))

# Last 24 full months window
src["month_start"] = src["Filing Date"].values.astype("datetime64[M]")
start_24 = last_full_month - pd.DateOffset(months=23)
win = src[(src["month_start"] >= start_24) & (src["month_start"] <= last_full_month)].copy()
win["mmYYYY"] = win["month_start"].dt.strftime("%m/%Y")

# Monthly counts by job type
m_job = (
    win.groupby(["month_start", "mmYYYY", "Job Type"], observed=False)
       .size().reset_index(name="filings")
)

# Keep top 6 job types across the window for readability
top6 = (
    m_job.groupby("Job Type", observed=False)["filings"].sum()
         .sort_values(ascending=False).head(6).index.tolist()
)
m_job_top = m_job[m_job["Job Type"].isin(top6)].copy()

# Order months on x-axis
month_order = (
    m_job_top[["month_start","mmYYYY"]]
    .drop_duplicates()
    .sort_values("month_start")["mmYYYY"]
    .tolist()
)

# Overview series (total filings per month, all job types) for brush selector
m_overview = (
    m_job_top.groupby(["month_start","mmYYYY"], observed=False)["filings"]
             .sum().reset_index(name="total_filings")
).sort_values("month_start")

#  Interactions
legend_sel = alt.selection_point(fields=["Job Type"], bind="legend", toggle="true")  # click legend to (de)select
brush = alt.selection_interval(encodings=["x"])  # brush over months in the overview to filter main

#  Main clustered bars
bars = (
    alt.Chart(m_job_top)
    .mark_bar()
    .encode(
        x=alt.X("mmYYYY:N",
                title="Month (MM/YYYY)",
                sort=month_order,
                axis=alt.Axis(labelAngle=-40)),
        y=alt.Y("filings:Q", title="Filings (count)"),
        color=alt.Color("Job Type:N", title="Job Type"),
        xOffset=alt.XOffset("Job Type:N"),
        order=alt.Order("Job Type:N"),
        opacity=alt.condition(legend_sel, alt.value(1.0), alt.value(0.2)),
        tooltip=[
            alt.Tooltip("mmYYYY:N", title="Month"),
            alt.Tooltip("Job Type:N"),
            alt.Tooltip("filings:Q", title="Filings")
        ]
    )
)

#  3-month rolling mean line (per Job Type) layered on top
# Uses month_start for correct temporal sorting; x still displays mmYYYY for nice ticks.
rolling_line = (
    alt.Chart(m_job_top)
    .transform_window(
        sort=[{"field": "month_start"}],
        groupby=["Job Type"],
        rolling_mean="mean(filings)",
        frame=[-2, 0]  # 3-month trailing average
    )
    .mark_line(point=True)
    .encode(
        x=alt.X("mmYYYY:N", sort=month_order, title=""),
        y=alt.Y("rolling_mean:Q", title=""),
        color=alt.Color("Job Type:N", title="Job Type"),
        strokeDash=alt.condition(legend_sel, alt.value([1,0]), alt.value([3,3])),
        opacity=alt.condition(legend_sel, alt.value(1.0), alt.value(0.3)),
        tooltip=[
            alt.Tooltip("mmYYYY:N", title="Month"),
            alt.Tooltip("Job Type:N"),
            alt.Tooltip("rolling_mean:Q", title="3-mo avg", format=",.1f")
        ]
    )
)

main = (
    (bars + rolling_line)
    .transform_filter(brush)           # filtered by bottom brush
    .add_params(legend_sel)            # legend highlight
    .properties(
        title="Monthly Filings by Job Type — Last 24 Full Months (Interactive)",
        width=900,
        height=360
    )
    .interactive(bind_y=True)
)

# Overview with brush to filter the main view
overview = (
    alt.Chart(m_overview)
    .mark_area(opacity=0.3)
    .encode(
        x=alt.X("mmYYYY:N", sort=month_order, title="Select month range"),
        y=alt.Y("total_filings:Q", title="Total filings"),
        tooltip=[alt.Tooltip("mmYYYY:N", title="Month"),
                 alt.Tooltip("total_filings:Q", title="Total")]
    )
    .add_params(brush)
    .properties(width=900, height=80)
)

# Compose
chart_interactive = alt.vconcat(main, overview).resolve_scale(x="shared")

chart_interactive


In [ ]:
#  Clustered bars + legend highlight + 3-mo rolling mean, with YEAR filter (2021–2025)

import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()


DF_local = DF.copy()

src = (
    DF_local[["Filing Date", "Job Type"]]
    .dropna(subset=["Filing Date", "Job Type"])
    .copy()
)

# Parse date + parts
src["Filing Date"] = pd.to_datetime(src["Filing Date"], errors="coerce")
src = src.dropna(subset=["Filing Date"]).copy()
src["year"]        = src["Filing Date"].dt.year.astype(int)
src["month_num"]   = src["Filing Date"].dt.month.astype(int)
src["month_start"] = src["Filing Date"].values.astype("datetime64[M]")
src["month_lbl"]   = src["Filing Date"].dt.strftime("%b")  # Jan, Feb, ...

# Limit to 2021–2025 if you want to bound the selector
src = src[(src["year"] >= 2021) & (src["year"] <= 2025)].copy()

# Monthly counts by job type
m_job = (
    src.groupby(["year", "month_num", "month_start", "month_lbl", "Job Type"], observed=False)
       .size()
       .reset_index(name="filings")
)

# Keep top 6 Job Types across ALL years (stable legend across selections)
top6 = (
    m_job.groupby("Job Type", observed=False)["filings"]
         .sum()
         .sort_values(ascending=False)
         .head(6)
         .index.tolist()
)
m_job_top = m_job[m_job["Job Type"].isin(top6)].copy()

# Month order: 1..12
month_order = list(range(1, 13))

# Year options and default
year_options = sorted(m_job_top["year"].unique().tolist())
default_year = max(year_options) if year_options else 2025


# Interactions

legend_sel = alt.selection_point(fields=["Job Type"], bind="legend", toggle="true")
year_sel = alt.selection_point(
    fields=["year"],
    bind=alt.binding_select(options=year_options, name="Year: "),
    value={"year": default_year}
)

# Axis with month abbreviations via labelExpr (no labelExprSignal)
axis_month = alt.Axis(
    title="Month",
    labelAngle=-40,
    # index into an array by the numeric tick value (1..12)
    labelExpr='["","Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"][toNumber(datum.value)]'
)

# Main clustered bars

bars = (
    alt.Chart(m_job_top)
    .transform_filter(year_sel)
    .mark_bar()
    .encode(
        x=alt.X("month_num:O", title="Month", sort=month_order, axis=axis_month),
        y=alt.Y("filings:Q", title="Filings (count)"),
        color=alt.Color("Job Type:N", title="Job Type"),
        xOffset=alt.XOffset("Job Type:N"),
        order=alt.Order("Job Type:N"),
        opacity=alt.condition(legend_sel, alt.value(1.0), alt.value(0.2)),
        tooltip=[
            alt.Tooltip("year:N", title="Year"),
            alt.Tooltip("month_lbl:N", title="Month"),
            alt.Tooltip("Job Type:N"),
            alt.Tooltip("filings:Q", title="Filings")
        ]
    )
)


# 3-month rolling mean line (computed within the selected year)

rolling_line = (
    alt.Chart(m_job_top)
    .transform_filter(year_sel)
    .transform_window(
        sort=[{"field": "month_num"}],
        groupby=["Job Type", "year"],
        rolling_mean="mean(filings)",
        frame=[-2, 0]  # trailing 3-month average
    )
    .mark_line(point=True)
    .encode(
        x=alt.X("month_num:O", sort=month_order, title="", axis=axis_month),
        y=alt.Y("rolling_mean:Q", title=""),
        color=alt.Color("Job Type:N", title="Job Type"),
        strokeDash=alt.condition(legend_sel, alt.value([1,0]), alt.value([3,3])),
        opacity=alt.condition(legend_sel, alt.value(1.0), alt.value(0.3)),
        tooltip=[
            alt.Tooltip("year:N", title="Year"),
            alt.Tooltip("month_lbl:N", title="Month"),
            alt.Tooltip("Job Type:N"),
            alt.Tooltip("rolling_mean:Q", title="3-mo avg", format=",.1f")
        ]
    )
)

chart = (
    (bars + rolling_line)
    .add_params(legend_sel, year_sel)
    .properties(
        title="Monthly Filings by Job Type — Select a Year",
        width=900,
        height=360
    )
    .interactive(bind_y=True)
)

chart


In [ ]:
# Linked View: (A) Monthly clustered bars + (B) Cost × FloorArea heatmap by Borough
import pandas as pd
import numpy as np
import altair as alt

alt.data_transformers.disable_max_rows()

# ---------------- Load / prep ----------------
DF_local = DF.copy()

# Minimal fields for both charts
D = DF_local[[
    "Filing Date", "Job Type", "Borough",
    "Initial Cost", "Total Construction Floor Area"
]].rename(columns={
    "Initial Cost": "InitialCost",
    "Total Construction Floor Area": "FloorArea"
}).dropna(subset=["Filing Date", "Job Type", "Borough"]).copy()

# Parse date parts
D["Filing Date"] = pd.to_datetime(D["Filing Date"], errors="coerce")
D = D.dropna(subset=["Filing Date"]).copy()
D["year"]      = D["Filing Date"].dt.year.astype(int)
D["month_num"] = D["Filing Date"].dt.month.astype(int)
D["month_lbl"] = D["Filing Date"].dt.strftime("%b")

# Numeric clean (keep positive values for log view)
D["InitialCost"] = pd.to_numeric(D["InitialCost"], errors="coerce")
D["FloorArea"]   = pd.to_numeric(D["FloorArea"], errors="coerce")
D = D[(D["InitialCost"] > 0) & (D["FloorArea"] > 0)].copy()

# Bound to assignment range if desired
D = D[(D["year"] >= 2021) & (D["year"] <= 2025)].copy()

# Monthly counts by job type for Chart A
m_job = (
    D.groupby(["year", "month_num", "month_lbl", "Job Type"], observed=False)
     .size().reset_index(name="filings")
)

# Keep top 6 Job Types across all years for stable legend
top6 = (m_job.groupby("Job Type", observed=False)["filings"]
             .sum().sort_values(ascending=False).head(6).index.tolist())
m_job_top = m_job[m_job["Job Type"].isin(top6)].copy()

# Interaction params (shared across charts)
year_options = sorted(m_job_top["year"].unique().tolist())
default_year = max(year_options) if year_options else 2025

legend_sel = alt.selection_point(fields=["Job Type"], bind="legend", toggle="true")
year_sel   = alt.selection_point(fields=["year"],
                                 bind=alt.binding_select(options=year_options, name="Year: "),
                                 value={"year": default_year})
# NEW: click months on the bars to filter the heatmap (toggle; double-click background clears)
month_sel = alt.selection_point(fields=["month_num"], on="click", toggle="true", empty=True)

axis_month = alt.Axis(
    title="Month",
    labelAngle=-40,
    labelExpr='["","Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"][toNumber(datum.value)]'
)

#  (A) Monthly clustered bars + 3-mo rolling mean
bars = (
    alt.Chart(m_job_top)
    .transform_filter(year_sel)
    .mark_bar()
    .encode(
        x=alt.X("month_num:O", sort=list(range(1,13)), axis=axis_month),
        y=alt.Y("filings:Q", title="Filings (count)"),
        color=alt.Color("Job Type:N", title="Job Type"),
        xOffset=alt.XOffset("Job Type:N"),
        order=alt.Order("Job Type:N"),
        # legend highlight
        opacity=alt.condition(legend_sel, alt.value(1.0), alt.value(0.2)),
        # show selection with a subtle outline
        stroke=alt.value("black"),
        strokeOpacity=alt.condition(month_sel, alt.value(0.9), alt.value(0)),
        tooltip=[
            alt.Tooltip("year:N", title="Year"),
            alt.Tooltip("month_lbl:N", title="Month"),
            alt.Tooltip("Job Type:N"),
            alt.Tooltip("filings:Q", title="Filings")
        ]
    )
    .add_params(month_sel)  # clicking bars toggles selected months
)

rolling = (
    alt.Chart(m_job_top)
    .transform_filter(year_sel)
    .transform_window(
        sort=[{"field": "month_num"}],
        groupby=["Job Type","year"],
        rolling_mean="mean(filings)",
        frame=[-2, 0]
    )
    .mark_line(point=True)
    .encode(
        x=alt.X("month_num:O", sort=list(range(1,13)), axis=axis_month, title=""),
        y=alt.Y("rolling_mean:Q", title=""),
        color=alt.Color("Job Type:N", title="Job Type"),
        strokeDash=alt.condition(legend_sel, alt.value([1,0]), alt.value([3,3])),
        opacity=alt.condition(legend_sel, alt.value(1.0), alt.value(0.3)),
        tooltip=[
            alt.Tooltip("year:N", title="Year"),
            alt.Tooltip("month_lbl:N", title="Month"),
            alt.Tooltip("Job Type:N"),
            alt.Tooltip("rolling_mean:Q", title="3-mo avg", format=",.1f")
        ]
    )
)

chart_A = (
    (bars + rolling)
    .add_params(legend_sel, year_sel)
    .properties(title="(A) Monthly Filings by Job Type — Select Year; Click Months to Link",
                width=900, height=340)
    .interactive(bind_y=True)
)

# (B) Cost × FloorArea density heatmap, faceted by Borough
# Log10 transforms for stable binning (avoid log scales on binned axes)
D_heat = D.copy()
D_heat["floor_log"] = np.log10(D_heat["FloorArea"])
D_heat["cost_log"]  = np.log10(D_heat["InitialCost"])

axis_log_fa = alt.Axis(
    title="Floor Area (sq ft, log10)",
    labelExpr='format(pow(10, datum.value), ",.0s")'  # 1k, 10k, 100k...
)
axis_log_cost = alt.Axis(
    title="Initial Cost (USD, log10)",
    labelExpr='format(pow(10, datum.value), ",.0s")'
)

heat = (
    alt.Chart(D_heat)
    # Link to other interactions
    .transform_filter(year_sel)
    .transform_filter(legend_sel)   # respects legend job-type filter
    .transform_filter(month_sel)    # respects clicked months; empty => all months
    .mark_rect()
    .encode(
        x=alt.X("floor_log:Q", bin=alt.Bin(step=0.25), axis=axis_log_fa),
        y=alt.Y("cost_log:Q",  bin=alt.Bin(step=0.25), axis=axis_log_cost),
        color=alt.Color("count():Q", title="Applications", legend=alt.Legend(format="~s")),
        tooltip=[
            alt.Tooltip("Borough:N"),
            alt.Tooltip("count():Q", title="Apps"),
        ]
    )
    .properties(width=170, height=150)
)

# Facet by Borough
borough_order = ["Manhattan","Brooklyn","Queens","Bronx","Staten Island"]
chart_B = (
    heat.transform_filter("isValid(datum.Borough)")
        .facet(column=alt.Column("Borough:N", sort=borough_order, title="Borough"))
        .resolve_scale(color="independent")  # per-borough color normalization
        .properties(title="(B) Cost × Floor Area — Linked to Year/Month/Job Type")
)

# Compose the linked view
linked_view = alt.vconcat(chart_A, chart_B).resolve_scale(color="independent")
linked_view


#### Final Visualization produced after the above iterative process


In [ ]:
#  LINKED VIEW: (A) Monthly clustered bars + (B) Work-Type mix by Borough
import pandas as pd
import altair as alt
import numpy as np

alt.data_transformers.disable_max_rows()

#  Load / base prep
DF_local = DF.copy()

# Minimal columns
base = DF_local[[
    "Filing Date", "Job Type", "Borough",
    # Work Type flags & related
    "Sprinkler (Work Type)", "Plumbing (Work Type)", "Boiler Equipment (Work Type)",
    "Earth Work (Work Type)", "Foundation (Work Type)", "General Construction (Work Type)",
    "Mechanical Systems (Work Type)", "Place of Assembly (Work Type)",
    "Protection Mechanical Methods (Work Type)", "Sidewalk Shed (Work Type)",
    "Structural (Work Type)", "Support of Excavation (Work Type)",
    "Temporary Place of Assembly (Work Type)"
]].copy()

# Parse dates
base["Filing Date"] = pd.to_datetime(base["Filing Date"], errors="coerce")
base = base.dropna(subset=["Filing Date", "Job Type", "Borough"]).copy()
base["year"]      = base["Filing Date"].dt.year.astype(int)
base["month_num"] = base["Filing Date"].dt.month.astype(int)
base["month_lbl"] = base["Filing Date"].dt.strftime("%b")


base = base[(base["year"] >= 2021) & (base["year"] <= 2025)].copy()

#
# 1) detect work-type columns you actually have (some may be missing)
worktype_cols = [c for c in base.columns if c.endswith("( Work Type)") or c.endswith("(Work Type)")]
worktype_cols = [c for c in worktype_cols if c not in ["Filing Date","Job Type","Borough"]]

# 2) coerce to boolean robustly (handles True/False, 1/0, Y/N, strings)
def to_bool_series(s):
    if s.dtype == bool:
        return s.fillna(False)
    if pd.api.types.is_numeric_dtype(s):
        return s.fillna(0).astype(int).astype(bool)
    # strings / object
    sv = s.astype(str).str.strip().str.lower()
    return sv.isin(["y","yes","true","t","1"])

for c in worktype_cols:
    base[c] = to_bool_series(base[c])


longWT = base.melt(
    id_vars=["Filing Date","year","month_num","month_lbl","Job Type","Borough"],
    value_vars=worktype_cols,
    var_name="WorkTypeRaw", value_name="has_work"
)
longWT = longWT[longWT["has_work"]].copy()

# Clean label: remove " (Work Type)"
longWT["WorkType"] = longWT["WorkTypeRaw"].str.replace(r"\s*\( ?Work Type ?\)\s*", "", regex=True)

# Chart A: Monthly clustered bars (Job Type)
m_job = (
    base.groupby(["year","month_num","month_lbl","Job Type"], observed=False)
        .size().reset_index(name="filings")
)

# Keep top N job types across all years for a stable legend
topN = 6
top_jobs = (m_job.groupby("Job Type", observed=False)["filings"]
                 .sum().sort_values(ascending=False).head(topN).index.tolist())
m_job_top = m_job[m_job["Job Type"].isin(top_jobs)].copy()

# Interactions
year_options = sorted(m_job_top["year"].unique().tolist())
default_year = max(year_options) if year_options else 2025

legend_sel = alt.selection_point(fields=["Job Type"], bind="legend", toggle="true")
year_sel   = alt.selection_point(
    fields=["year"],
    bind=alt.binding_select(options=year_options, name="Year: "),
    value={"year": default_year}
)
brush_months = alt.selection_interval(encodings=["x"], empty=True)  # brush across months on chart A

axis_month = alt.Axis(
    title="Month",
    labelAngle=-40,
    labelExpr='["","Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"][toNumber(datum.value)]'
)

bars = (
    alt.Chart(m_job_top)
    .transform_filter(year_sel)
    .mark_bar()
    .encode(
        x=alt.X("month_num:O", sort=list(range(1,13)), axis=axis_month),
        y=alt.Y("filings:Q", title="Filings (count)"),
        color=alt.Color("Job Type:N", title="Job Type"),
        xOffset=alt.XOffset("Job Type:N"),
        order=alt.Order("Job Type:N"),
        opacity=alt.condition(legend_sel, alt.value(1.0), alt.value(0.25)),
        tooltip=[
            alt.Tooltip("year:N", title="Year"),
            alt.Tooltip("month_lbl:N", title="Month"),
            alt.Tooltip("Job Type:N"),
            alt.Tooltip("filings:Q", title="Filings")
        ]
    )
    .add_params(brush_months)
)


rolling = (
    alt.Chart(m_job_top)
    .transform_filter(year_sel)
    .transform_window(
        sort=[{"field": "month_num"}],
        groupby=["Job Type","year"],
        rolling_mean="mean(filings)",
        frame=[-2, 0]
    )
    .mark_line(point=True)
    .encode(
        x=alt.X("month_num:O", sort=list(range(1,13)), axis=axis_month, title=""),
        y=alt.Y("rolling_mean:Q", title=""),
        color=alt.Color("Job Type:N", title="Job Type"),
        strokeDash=alt.condition(legend_sel, alt.value([1,0]), alt.value([3,3])),
        opacity=alt.condition(legend_sel, alt.value(1.0), alt.value(0.35)),
        tooltip=[
            alt.Tooltip("year:N", title="Year"),
            alt.Tooltip("month_lbl:N", title="Month"),
            alt.Tooltip("Job Type:N"),
            alt.Tooltip("rolling_mean:Q", title="3-mo avg", format=",.1f")
        ]
    )
)

chart_A = (
    (bars + rolling)
    .add_params(legend_sel, year_sel)
    .properties(
        title="(A) Monthly Filings by Job Type — Year selector + Brush months to link",
        width=1000, height=500
    )
    .interactive(bind_y=True)
)

#  Chart B: Work-Type mix by Borough (linked)
# Filter by: selected year, legend (Job Type), and brushed months from chart A
worktype_mix = (
    alt.Chart(longWT)
    .transform_filter(year_sel)
    .transform_filter(legend_sel)
    .transform_filter(brush_months)
    .mark_bar()
    .encode(
        y=alt.Y("Borough:N", sort=["Manhattan","Brooklyn","Queens","Bronx","Staten Island"], title="Borough"),
        x=alt.X("count():Q", stack="normalize", title="Share of filings"),
        color=alt.Color("WorkType:N", title="Work Type"),
        tooltip=[
            alt.Tooltip("Borough:N"),
            alt.Tooltip("WorkType:N", title="Work Type"),
            alt.Tooltip("count():Q", title="Filings", format=",.0f")
        ]
    )
    .properties(title="(B) Work-Type Composition by Borough — Linked to (A)", width=900, height=220)
)

monthly_jobtype_boro = (
    base.groupby(["year", "month_num", "Borough", "Job Type"], observed=False)
        .size().reset_index(name="filings")
)

BORO_DOMAIN = ["Manhattan","Brooklyn","Queens","Bronx","Staten Island"]

heatmap_C = (
    alt.Chart(monthly_jobtype_boro, title="(C) Borough × Job Type — Filings Heatmap (linked)")
      .transform_filter(year_sel)         # dropdown year
      .transform_filter(brush_months)     # brushed months from chart A
      .transform_filter(legend_sel)       # respects Job Type legend toggle
      # keep the same Job Types as chart A's legend (topN)
      .transform_filter(alt.FieldOneOfPredicate(field="Job Type", oneOf=top_jobs))
      # aggregate over the brushed months -> single value per Borough × Job Type
      .transform_aggregate(
          filings="sum(filings)",
          groupby=["Borough", "Job Type"]
      )
      .mark_rect()
      .encode(
          x=alt.X("Job Type:N", title="Job Type", sort=top_jobs,
                  axis=alt.Axis(labelAngle=-45)),
          y=alt.Y("Borough:N", title="Borough", sort=BORO_DOMAIN),
          color=alt.Color(
              "filings:Q", title="Filings",
              scale=alt.Scale(type="log", domainMin=1, scheme="blues")
          ),
          tooltip=[
              alt.Tooltip("Borough:N"),
              alt.Tooltip("Job Type:N"),
              alt.Tooltip("filings:Q", title="Filings", format=",")
          ]
      )
      .properties(width=440, height=220)
)

#  Composition
linked = alt.vconcat(
    chart_A,
    alt.hconcat(worktype_mix, heatmap_C)
).resolve_scale(color="independent")

linked